# Notebook with code snippets for (simple) running HBV-SASK model

## Originally, data comes from this source: TBA

## This notebook contains code snippets for simply running the model and examing / plotting the output

In [ ]:
import numpy as np
import pathlib
import pandas as pd
import sys
import time

In [ ]:
# TODO - change this path accordingly
# sys.path.insert(1, '/work/ga45met/Hydro_Models/HBV-SASK-py-tool')
sys.path.insert(1, '/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic')

In [ ]:
from uqef_dynamic.utils import utility
from uqef_dynamic.models.hbv_sask import hbvsask_utility as hbv
from uqef_dynamic.models.hbv_sask import HBVSASKModel as hbvmodel

In [ ]:
# importing modules/libs for plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

import matplotlib.pyplot as mp

from plotly.offline import plot

pd.options.plotting.backend = "plotly"

### Defining paths

In [ ]:
# TODO - change these paths accordingly
hbv_model_data_path = pathlib.Path("/work/ga45met/Hydro_Models/HBV-SASK-data")
configurationObject = pathlib.Path('/work/ga45met/Hydro_Models/HBV-SASK-py-tool/configurations/configuration_hbv_6D.json')
# configurationObject = pathlib.Path('/work/ga45met/Hydro_Models/HBV-SASK-py-tool/configurations/configuration_hbv_10D_full.json')
# configurationObject = pathlib.Path('/work/ga45met/Hydro_Models/HBV-SASK-py-tool/configurations/configuration_hbv_10D_MC_banff.json')
# configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Hydro/configurations/configuration_hbv_6D.json')
basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'

inputModelDir = hbv_model_data_path

# TODO - change this path accordingly
workingDir = hbv_model_data_path / basis / "model_runs" / 'ensamble_run_full' #"whole_time_generating_state_df"


# Creating Model Object

Creating a model object

In [ ]:
writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

In [ ]:
# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
hbvsaskModelObject.start_date

Examing the input/forcing data and ground-truth/measured data if on disposal...

In [ ]:
hbvsaskModelObject.time_series_measured_data_df

In [ ]:
hbvsaskModelObject.time_series_measured_data_df['streamflow'].values #y_hat

In [ ]:
hbvsaskModelObject.plot_input_data(read_measured_streamflow=True)

In [ ]:
hbvsaskModelObject.initial_condition_df

## Analysing initial condition file

In [ ]:
hbvsaskModelObject.initial_condition_file

In [ ]:
temp = hbv.read_initial_conditions(initial_condition_file=hbvsaskModelObject.initial_condition_file)
temp

In [ ]:
temp = hbv.read_initial_conditions(
    hbvsaskModelObject.initial_condition_file, 
    timestamp=hbvsaskModelObject.start_date,
    time_column_name=hbvsaskModelObject.time_column_name
)
temp

In [ ]:
# basis = "Banff_Basin" #"Oldman_Basin"
temp_initial_condition_file = hbv_model_data_path / basis / "state_df.pkl"
temp_initial_condition_file_const = hbv_model_data_path / basis / "state_const_df.pkl"

temp  = pd.read_pickle(temp_initial_condition_file, compression="gzip")
temp

In [ ]:
temp  = pd.read_pickle(temp_initial_condition_file_const, compression="gzip")
temp

In [ ]:
temp  = pd.read_pickle(temp_initial_condition_file, compression="gzip")
print(f"temp - after saving it again {temp}")

# Running a single model run without changing the parameter values

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

## Examing the model output

Model run returns an array. \
Each element of the array is a tuple \
The first element of each tuple is a dictionary storing different info about model run \
The second element of each tuple is a runtime (which is as well stored in the above-mentioned dictionary)

In [ ]:
print(type(results_array))
print(f"len of the resulted array is equal to the total number of model runs (one set of parameters one run) - {len(results_array)}")
print(type(results_array[0]))
print(f"The first element of each tuple is a {type(results_array[0][0])}")
print(f"The second element of each tuple is a {type(results_array[0][1])}")
print(f"runtime : {results_array[0][1]}")

In [ ]:
print(f"start_date - {hbvsaskModelObject.start_date}")
print(f"end_date - {hbvsaskModelObject.end_date}")
print(f"start_date_predictions - {hbvsaskModelObject.start_date_predictions}")


In [ ]:
results_array[0]

more about the result dictionary... it stores different dataframes

In [ ]:
results_array[0][0].keys()

In [ ]:
results_array[0][0]['run_time']

In [ ]:
results_array[0][0]['parameters_dict']

pd.DataFrame storing index_run, paramter values and values for different likelihood functions/goodness-of-fit (GoF) functions

In [ ]:
results_array[0][0]['gof_df']

the output of the model is in the form of a time-series stored in a pd.DataFrame...

In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
results_array[0][0]['result_time_series'].columns

plotting input, predicted/simulated and measured time-series

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

column 'stramflow' contains the measured data, column "Q_cms" contains predicted data (i.e., streamflow expressed in cubic meters per second) by the model defined with the current values for the uncertain parameters

as a QoI, one can take Q_cms time-series (extract it from above dataframe), AET (Actual EvapoTranspiration), or some likelihood (i.e., goodness-of-fit (GoF)) function value

In [ ]:
qoi = results_array[0][0]['result_time_series']["Q_cms"].values
qoi

In [ ]:
qoi_2 = results_array[0][0]['gof_df']["RMSE"].values
qoi_2

In [ ]:
# plotting state
state_df = results_array[0][0]['state_df']
fig = go.Figure()
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["SMS"],name="Soil Storage",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S1"],name="Fast Reservoir",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S2"],name="Slow Reservoir",))
fig.show()

In [ ]:
# Note: the utility function plotting_input_output_state is defined at the bottom of the notebook
# fig = plotting_input_output_state(results_array)
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject,
    result_df = results_array[0][0]['result_time_series'],
    state_df = results_array[0][0]['state_df']
)
fig.show()

# Running a single model run with propagating the values for uncertain parameters specified in json configuration file

One can specify the direct values of the uncertain parameters in the form of a dictionary. The order and naming of the parameters has to follow the order from the configuration_json_file_dict["parameters"] 

In [ ]:
parameter_value_dict = {'TT': -4.0, 'C0': 0.0, 'ETF': 0.0, 'FC': 50, 'FRAC': 0.1, 'K2': 0.025} # old default values
parameter_value_dict_1 = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.1, 'K2': 0.025}
parameter_value_dict_2 = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.1, 'K2': 0.1}
parameter_value_dict_3 = {'TT': -4.0, 'C0': 5.0, 'ETF': 0.5, 'FC': 50, 'FRAC': 0.1, 'K2': 0.025}
parameter_value_dict = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.3, 'K2': 0.1, 'K1': 0.5, 'alpha':2.0}



In [ ]:
unique_run_index = 1 
start = time.time()
results_array_changed_param = hbvsaskModelObject.run(
#     i_s = [1,2,3,4],
#     parameters = [parameter_value_dict, parameter_value_dict_1, parameter_value_dict_2, parameter_value_dict_3],
    i_s = [unique_run_index,],
    parameters = [parameter_value_dict,],
    createNewFolder=createNewFolder,
    take_direct_value=True
)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
results_array_changed_param[0][0]['parameters_dict']

In [ ]:
results_array_changed_param[0][0]['result_time_series']

In [ ]:
results_array_changed_param[0][0]['result_time_series']['Q_cms'].values #y

In [ ]:
results_array_changed_param[0][0]['state_df']

## Examing the model output

In [ ]:
results_array_changed_param[0][0]['parameters_dict']

In [ ]:
results_array_changed_param[0][0]['result_time_series']

In [ ]:
# dataframe containing predicted state data
state_df = results_array_changed_param[0][0]['state_df']
state_df

Compare the 'newly' computed state DataFrame with the one saved for the whole time span for this basin...

In [ ]:
temp_initial_condition_file = hbv_model_data_path / basis / "state_df.pkl"
temp  = pd.read_pickle(temp_initial_condition_file, compression="gzip")
temp

In [ ]:
fig = px.line(state_df, x=state_df.index, y=['SWE',], title="SWE")
fig.show()

In [ ]:
fig = px.line(state_df, x=state_df.index, y=['S2',], title="S2")
fig.show()

In [ ]:
# fig = state_df.plot(x=state_df.index, y=["SMS", "S1", "S2"], kind="line", )

fig = go.Figure()
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["SMS"],name="Soil Storage",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S1"], name="Fast Reservoir",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S2"], name="Slow Reservoir",))
fig.show()

In [ ]:
min_time_index = results_array_changed_param[0][0]['result_time_series'].index.min()
max_time_index = results_array_changed_param[0][0]['result_time_series'].index.max()
print(min_time_index)
print(max_time_index)

print(hbvsaskModelObject.time_series_measured_data_df.index.min())
print(hbvsaskModelObject.time_series_measured_data_df.index.max())

parsed_input_data_df = hbvsaskModelObject.time_series_measured_data_df.loc[min_time_index:max_time_index]

print(parsed_input_data_df.index.min())
print(parsed_input_data_df.index.max())

In [ ]:
parsed_input_data_df

In [ ]:
error_time_series = np.array(parsed_input_data_df['streamflow'].values) - \
np.array(results_array_changed_param[0][0]['result_time_series']['Q_cms'].values)
error_time_series



# Plotting the output and input data

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array_changed_param[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

In [ ]:
hbvsaskModelObject.time_series_measured_data_df.columns

In [ ]:
results_array_changed_param[0][0]['result_time_series'].columns

In [ ]:
results_array_changed_param[0][0]['state_df'].columns

In [ ]:
# fig = hbv.plot_input_output_state(
#     modelObject = hbvsaskModelObject,
#     result_df = results_array_changed_param[0][0]['result_time_series'],
#     state_df = results_array_changed_param[0][0]['state_df']
# )

fig = plot_input_output_state(modelObject=hbvsaskModelObject, 
                              result_df=results_array_changed_param[0][0]['result_time_series'], 
                              state_df=results_array_changed_param[0][0]['state_df'])
fig.show()

# Reading the saved output of the model

In [ ]:
# paths to saved output from model run
path_to_input = hbv_model_data_path / basis
i = 1 # index of model run of interest
if createNewFolder:
    flux_output_file = workingDir / f"run_{i}" / f"flux_df_{i}.pkl"
    state_output_file = workingDir / f"run_{i}" / f"state_df_{i}.pkl"
else:
    flux_output_file = workingDir / f"flux_df_{i}.pkl"
    state_output_file = workingDir / f"state_df_{i}.pkl"

In [ ]:
flux_df = pd.read_pickle(flux_output_file, compression="gzip")
flux_df

In [ ]:
state_df = pd.read_pickle(state_output_file, compression="gzip")
state_df

In [ ]:
hbvsaskModelObject.streamflow_column_name
hbvsaskModelObject.time_series_measured_data_df

In [ ]:
# re-computation of some qoodnes-of-fit/likelihood functions
# flux_df = results_array_changed_param[0][0]['result_time_series']
gof_list = ["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"]
gof_dict = utility.calculateGoodnessofFit_simple(
    measuredDF = hbvsaskModelObject.time_series_measured_data_df,
    simulatedDF = flux_df,
    gof_list = gof_list,
    measuredDF_time_column_name=hbvsaskModelObject.time_column_name,
    measuredDF_column_name=hbvsaskModelObject.streamflow_column_name,
    simulatedDF_time_column_name=hbvsaskModelObject.time_column_name,
    simulatedDF_column_name='Q_cms',
    return_dict=True,
)
gof_dict

### Utility function for plotting

In [ ]:
# or more detailed plotting of precipitation and temperature as main input data
# predicted streamflow and measured one
# and state data...

def plot_input_output_state(modelObject, result_df, state_df):
    # result_df = results_array[0][0]['result_time_series']
    # state_df = results_array[0][0]['state_df']
    # parsed_input_data_df = hbvsaskModelObject.time_series_measured_data_df.loc[
    #     result_df.index.min():result_df.index.max()]
    parsed_input_data_df = modelObject.time_series_measured_data_df.loc[
        result_df.index.min():result_df.index.max()]

    fig = make_subplots(
        rows=6, cols=1,
        subplot_titles=("Temperature", "Precipitation", "Streamflow", "EvapoTranspiration", "Snow Storage", "Soil+Reservoirs")
    )

    fig.add_trace(
        go.Scatter(
            x=parsed_input_data_df.index, y=parsed_input_data_df['temperature'],
            text=parsed_input_data_df['temperature'], 
            name="Temperature"
        ), 
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=parsed_input_data_df.index, y=parsed_input_data_df['precipitation'],
            text=parsed_input_data_df['precipitation'], 
            name="Precipitation"
        ), 
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=parsed_input_data_df.index, y=parsed_input_data_df['streamflow'],
            name="Observed Streamflow"
        ),
        row=3, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=result_df.index, y=result_df['Q_cms'],
            text=result_df['Q_cms'], 
            name="Predicted Streamflow"
        ), 
        row=3, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=result_df.index, y=result_df['AET'],
            text=result_df['AET'], 
            name="AET"
        ), 
        row=4, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=result_df.index, y=result_df['PET'],
            text=result_df['PET'], 
            name="PET"
        ), 
        row=4, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=state_df.index, y=state_df['SWE'],
            text=state_df['SWE'], 
            name="Snow Storage"
        ), 
        row=5, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=state_df.index, y=state_df['SMS'],
            text=state_df['SMS'], 
            name="Soil Storage"
        ), 
        row=6, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=state_df.index, y=state_df['S1'],
            text=state_df['S1'], 
            name="Fast Reservoir"
        ), 
        row=6, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=state_df.index, y=state_df['S2'],
            text=state_df['S2'], 
            name="Slow Reservoir"
        ), 
        row=6, col=1
    )
    
    fig.update_layout(
        height=1000, width=800, 
        #title_text="Detailed plot of most important time-series"
    )
    
    # fig.update_layout(
    #     title={
    #         'y': 1.00,  # Vertical position (increase to move title up)
    #         'x': 0.5,   # Horizontal position (0.5 centers it)
    #         'xanchor': 'center',  # Title's x-anchor position (to align the title horizontally)
    #         'yanchor': 'top'      # Title's y-anchor position (to align the title vertically)
    #     }
    # )

    fig.update_layout(
        # legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99),
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1), # y=1.02
        #title=f'HBV-SASK Model: Predicted vs. Observed Streamflow',
        showlegend=True,
        # template="plotly_white",
    )
    return fig

## Experimenting with different QoIs

### Experiment 1 - Sliding Window GoF

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":"GoF",
    "qoi_column":"Q_cms",
    "autoregressive_model_first_order":"False",
    "transform_model_output":"None",
    "read_measured_data": "True",
    "qoi_column_measured":"streamflow",
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"sliding_window",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"False",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "False",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")


In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

### Experiment 2 - Multiple QoI with transformation

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":["Q_cms","AET"],
    "qoi_column":["Q_cms","AET"],
    "autoregressive_model_first_order":"False",
    "transform_model_output":["log", "None"],
    "read_measured_data": ["True","False"],
    "qoi_column_measured":["streamflow","None"],
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"continuous",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"False",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "False",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
results_array[0][0]['result_time_series'].columns

In [ ]:
hbvsaskModelObject.time_series_measured_data_df

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

## Experiment 3 - gradient computation

In [ ]:
#import json

#configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_6D.json')

basis = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'
inputModelDir = hbv_model_data_path
workingDir = hbv_model_data_path / basis / "model_runs" / 'trying_out_stuff'

# with open(configurationObject) as f:
#     configurationObject = json.load(f)
configurationObject = {
    "time_settings":
    {
      "start_day": 1,
      "start_month": 9,
      "start_year": 2003,
      "start_hour": 0,
      "start_minute": 0,

      "end_day": 10,
      "end_month": 11,
      "end_year": 2007,
      "end_hour": 0,
      "end_minute": 0,

      "run_full_timespan":"False",
      "spin_up_length":1095,
      "simulation_length": 364,
      "resolution": "daily",

      "cut_runs": "False",
      "timestep": 5
    },
  "model_settings": {
    "basis": "Oldman_Basin",
    "plotting": "False",
    "writing_results_to_a_file": "False",
    "corrupt_forcing_data": "False"
  },
  "model_paths": {
    "hbv_model_path": "Hydro_Models/HBV-SASK-data"
  },
  "simulation_settings": {
    "qoi":["Q_cms","AET"],
    "qoi_column":["Q_cms","AET"],
    "autoregressive_model_first_order":"False",
    "transform_model_output":["None", "None"],
    "read_measured_data": ["True","False"],
    "qoi_column_measured":["streamflow","None"],
    "objective_function_qoi":"RMSE",
    "calculate_GoF":"False",
    "objective_function":["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"],
    "mode":"continuous",
    "interval": 365,
    "min_periods": 365,
    "method": "avrg",
    "center": "center",
    "compute_gradients":"True",
    "eps_gradients": 0.01,
    "gradient_method":"Forward Difference",
    "gradient_analysis": "True",
    "compute_active_subspaces": "True",
    "save_gradient_related_runs": "True"
  },
  "parameters": [
    {
      "name": "beta",
      "distribution": "Uniform",
      "lower": 1.0,
      "upper": 4.0,
      "default": 2.0,
      "values_list":[0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
    }
  ],
}    

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basis=basis,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(
    createNewFolder=createNewFolder,
)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
len(results_array)

In [ ]:
results_array[0][0]['result_time_series'].columns

In [ ]:
results_array[0][0]['parameters_dict']

In [ ]:
start = time.time()
list_of_parmeter_beta_values = [0.1, 0.5, 1.0, 2.0, 3.0, 4.0]
print(f"list_of_parmeter_beta_values - {type(list_of_parmeter_beta_values)}")
results_array = hbvsaskModelObject.run(
    i_s = range(0,len(list_of_parmeter_beta_values)),
    parameters = list_of_parmeter_beta_values,
    createNewFolder=createNewFolder,
    merge_output_with_measured_data=True
)
end = time.time()
runtime = end - start
print(f"#{len(list_of_parmeter_beta_values)}  execution(s) of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
results_array[0][0]['result_time_series'].columns

In [ ]:
len(results_array)

In [ ]:
type(results_array[0])

In [ ]:
type(results_array[0][0])

In [ ]:
results_array[0][0].keys()

In [ ]:
results_array[1][0]['result_time_series']

In [ ]:
results_array[1][0]['parameters_dict']['beta']

In [ ]:
fig = go.Figure()
for indx in range(0, len(results_array)):
    temp = results_array[indx][0]['result_time_series']
    fig.add_trace(go.Scatter(x=temp.index,y=temp["d_Q_cms_d_beta"], name=results_array[indx][0]['parameters_dict']['beta'],))
fig.show()

In [ ]:
fig = go.Figure()
for indx in range(0, len(results_array)):
    temp = results_array[indx][0]['result_time_series']
    fig.add_trace(go.Scatter(x=temp.index,y=temp["d_AET_d_beta"], name=results_array[indx][0]['parameters_dict']['beta'],))
fig.show()

In [ ]:
results_array[0][0]['parameters_dict']